# Qwen2.5-VL SFT Fine-Tuning (Kaggle GPU)

Supervised Fine-Tuning on PRM-verified correct reasoning trajectories.
**Goal**: Enforce structured step-by-step outputs, improve logical reasoning, reduce visual/arithmetic hallucinations.

In [ ]:
# === Cell 1: Environment Setup ===
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded.')
except Exception:
    print('Kaggle secrets unavailable.')

if not os.path.exists('prm_project'):
    !git clone https://github.com/yahorlahunovich/prm_project.git
%cd prm_project
!git pull --ff-only

import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'Compute Capability: {torch.cuda.get_device_capability(0)}')
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
# === Cell 2: Install Dependencies ===
import torch as _t
_tv = _t.__version__
print(f'Pinning torch=={_tv}')

!pip install -q \
    "torch=={_tv}" \
    "transformers>=4.49.0" \
    "trl>=0.12.0" \
    "peft>=0.10.0" \
    "accelerate>=0.30.0" \
    "datasets" \
    "qwen-vl-utils"

import transformers, trl, peft, accelerate
print(f'transformers={transformers.__version__}, trl={trl.__version__}, peft={peft.__version__}')

In [ ]:
# === Cell 3: Load Dataset ===
import json, os, torch
from datasets import Dataset
from PIL import Image

STRICT_ONLY = False  # True: 84 ultra-clean, False: ~300 GT-matching

images_dir = 'data/CharXiv/images'
if not os.path.exists(images_dir) or len(os.listdir(images_dir)) == 0:
    os.system('python scripts/download_images.py')

evals_path = 'experiments/001_500_reasoning/data/evaluated_rollouts.jsonl'
cleaned_path = 'experiments/001_500_reasoning/data/001_500_reasoning_cleaned.jsonl'

meta = {}
with open(cleaned_path) as f:
    for line in f:
        d = json.loads(line)
        qid, ridx = str(d['question_id']), d['rollout_index']
        gt = str(d['ground_truth']).strip().lower()
        ans = str(d['model_final_answer']).strip().lower()
        meta[(qid, ridx)] = {
            'is_correct': (gt in ans or ans in gt) and len(ans) > 0,
            'question': d.get('question', ''),
            'reasoning': d.get('reasoning_steps', '')
        }

sft_data = []
with open(evals_path) as f:
    for line in f:
        d = json.loads(line)
        qid, ridx = str(d['question_id']), d['rollout_index']
        evals = d.get('evaluations', [])
        if not evals: continue
        m = meta.get((qid, ridx), {})
        all_pass = all(s.get('score') == 1 for s in evals)
        keep = (all_pass and m.get('is_correct')) if STRICT_ONLY else m.get('is_correct')
        if keep and m.get('reasoning'):
            img_path = os.path.abspath(f'data/CharXiv/images/{qid}.jpg')
            if not os.path.exists(img_path): continue
            sft_data.append({'messages': [
                {'role': 'user', 'content': [
                    {'type': 'image', 'image': img_path},
                    {'type': 'text', 'text': f"Analyze this chart. Provide step-by-step reasoning and a final answer.\n{m['question']}"}
                ]},
                {'role': 'assistant', 'content': m['reasoning']}
            ]})

dataset = Dataset.from_list(sft_data)
print(f'Loaded {len(dataset)} SFT trajectories (STRICT_ONLY={STRICT_ONLY}).')

In [ ]:
# === Cell 4: Load Model & Processor ===
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

model_id = 'Qwen/Qwen2.5-VL-3B-Instruct'

if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    if cc[0] < 7:
        print(f'GPU {torch.cuda.get_device_name(0)} (cc {cc}) lacks PyTorch 2.10 sm_70+ support. Using CPU.')
        device_map = 'cpu'
    else:
        print(f'Using GPU {torch.cuda.get_device_name(0)} (cc {cc}).')
        device_map = {'': 0}
else:
    device_map = 'cpu'

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device_map != 'cpu' else torch.float32,
    attn_implementation='sdpa' if device_map != 'cpu' else 'eager',
    device_map=device_map,
)
model.enable_input_require_grads()
if hasattr(model, 'visual'): model.visual.requires_grad_(False)

processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=512*28*28)
processor.tokenizer.padding_side = 'right'
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.pad_token = processor.tokenizer.pad_token
processor.pad_token_id = processor.tokenizer.pad_token_id
processor.eos_token_id = processor.tokenizer.eos_token_id
print(f'Model loaded on device: {next(model.parameters()).device}')

In [ ]:
# === Cell 5: Configure & Train ===
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM',
)
is_gpu = next(model.parameters()).device.type == 'cuda'
training_args = SFTConfig(
    output_dir='./sft_qwen_vl', per_device_train_batch_size=1,
    gradient_accumulation_steps=4, learning_rate=2e-5, num_train_epochs=3,
    logging_steps=5, save_steps=50, save_total_limit=2,
    gradient_checkpointing=is_gpu, dataset_num_proc=1,
    remove_unused_columns=False, report_to='none',
    dataset_text_field='messages', fp16=is_gpu, bf16=False,
)

trainer = SFTTrainer(
    model=model, args=training_args, train_dataset=dataset,
    processing_class=processor, peft_config=peft_config,
)
trainer.train()
trainer.save_model('qwen_vl_sft_adapter')
print('SFT training complete! Adapter saved.')